# Adım 1: Docker Ortamının Kurulumu 
Bu aşamada Proje mimarisine uygun olarak Zookeeper, Kafka, Spark ve  Python Producer uygulamamız konteynerize edilmiştir.

## 1. docker-compose.yml Dosyası
Sistemi tek bir komutla ayağa kaldırmak ve servislerin birbiriyle  iletişim kurmasını sağlamak  için yazılan docker konfigürasyon dosyası:
```yaml
services:
  zookeeper:
    image: confluentinc/cp-zookeeper:7.3.0
    environment:
      ZOOKEEPER_CLIENT_PORT: 2181
      ZOOKEEPER_TICK_TIME: 2000
    healthcheck:
      test: ["CMD", "cub", "zk-ready", "localhost:2181", "10"]
      interval: 10s
      timeout: 5s
      retries: 5

  kafka:
    image: confluentinc/cp-kafka:7.3.0
    depends_on:
      zookeeper:
        condition: service_healthy
    ports:
      - "9092:9092"
    environment:
      KAFKA_BROKER_ID: 1
      KAFKA_ZOOKEEPER_CONNECT: zookeeper:2181
      KAFKA_LISTENERS: PLAINTEXT://0.0.0.0:9092
      KAFKA_ADVERTISED_LISTENERS: PLAINTEXT://kafka:9092
      KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR: 1
    healthcheck:
      test: ["CMD", "cub", "kafka-ready", "-b", "localhost:9092", "1", "20"]
      interval: 10s
      timeout: 5s
      retries: 5

  spark:
    build: ./spark
    user: root
    environment:
      - SPARK_MODE=master
      - HOME=/tmp
      - HADOOP_USER_NAME=root
    ports:
      - "8080:8080"
    volumes:
      - ./spark:/app/spark
      - ./data:/data
      - ./ml_pipeline:/app/ml_pipeline

  python-producer:
    build: ./producer
    depends_on:
      kafka:
        condition: service_healthy
    volumes:
      - ./data:/data

## 2. Python Producer İçin Dockerfile ve Gereksinimler
Kendi yazdığımız Producer uygulamasının bağımlılıkları (`requirements.txt`) ve Docker imajını oluşturduğumuz `Dockerfile` içeriği:

**requirements.txt:**
```text
kafka-python==2.0.2
pandas


## Dockerfile Producer Dosyası
```Dockerfile
FROM python:3.9-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY producer.py .
CMD ["python", "-u", "producer.py"]

## Dockerfile Spark Dosyası
```Dockerfile
FROM bitnamilegacy/spark:latest
USER root
RUN pip install pandas matplotlib seaborn mlflow scikit-learn

## 3. Çalıştırma
Sistemi ayağa kaldırmak için terminalde çalıştırdığımız komut:
`docker-compose up --build -d`